In [ ]:
# Ejecuta esta celda si recibes un error de ModuleNotFoundError (TensorFlow, etc.)
import sys
!{sys.executable} -m pip install tensorflow scikit-learn matplotlib numpy opencv-python

# Sistema de Clasificación de Imágenes: Perros vs Gatos

Este notebook implementa un clasificador de redes neuronales convolucionales (CNN) para gatos y perros basándose en la arquitectura VGG16. El conjunto de datos consta de 25,000 imágenes proporcionadas por Petfinder.com y Microsoft.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPool2D, Flatten, Dense
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from sklearn.model_selection import train_test_split

base_path = '../dogs-vs-cats/train/'
img_width, img_height = 200, 200

def load_dataset(path):
    images = []
    labels = []
    files = os.listdir(path)
    
    print(f"Cargando {len(files)} imágenes...")
    for i, file in enumerate(files):
        if i % 5000 == 0: print(f"Procesadas {i} imágenes...")
        label = 1 if 'dog' in file else 0
        
        img = load_img(os.path.join(path, file), target_size=(img_width, img_height))
        img_array = img_to_array(img)
        
        images.append(img_array)
        labels.append(label)
    
    return np.array(images), np.array(labels)

photos, labels = load_dataset(base_path)
print("Carga finalizada. Forma de los datos:", photos.shape)

### Paso 2: Visualiza la información de entrada

Vamos a visualizar las primeras nueve fotos de perros y de gatos.

In [ ]:
def plot_images(images, labels, title, class_idx):
    idx = np.where(labels == class_idx)[0][:9]
    plt.figure(figsize=(10, 10))
    for i, img_idx in enumerate(idx):
        plt.subplot(3, 3, i + 1)
        plt.imshow(images[img_idx].astype('uint8'))
        plt.axis('off')
    plt.suptitle(title)
    plt.show()

plot_images(photos, labels, "Primeras 9 fotos de Perros", 1)
plot_images(photos, labels, "Primeras 9 fotos de Gatos", 0)

Normalización y creación de generadores para entrenamiento y prueba.

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

photos = photos / 255.0
X_train, X_test, y_train, y_test = train_test_split(photos, labels, test_size=0.2, random_state=42)

datagen = ImageDataGenerator()
trdata = datagen.flow(X_train, y_train, batch_size=64)
tsdata = datagen.flow(X_test, y_test, batch_size=64)

### Paso 3: Construye una RNA (Arquitectura VGG16)

In [ ]:
from tensorflow.keras.layers import Conv2D, MaxPool2D, Flatten, Dense

model = Sequential()
model.add(Conv2D(input_shape = (200,200,3), filters = 64, kernel_size = (3,3), padding = "same", activation = "relu"))
model.add(Conv2D(filters = 64,kernel_size = (3,3),padding = "same", activation = "relu"))
model.add(MaxPool2D(pool_size = (2,2),strides = (2,2)))

model.add(Conv2D(filters = 128, kernel_size = (3,3), padding = "same", activation = "relu"))
model.add(Conv2D(filters = 128, kernel_size = (3,3), padding = "same", activation = "relu"))
model.add(MaxPool2D(pool_size = (2,2),strides = (2,2)))

model.add(Conv2D(filters = 256, kernel_size = (3,3), padding = "same", activation = "relu"))
model.add(Conv2D(filters = 256, kernel_size = (3,3), padding = "same", activation = "relu"))
model.add(Conv2D(filters = 256, kernel_size = (3,3), padding = "same", activation = "relu"))
model.add(MaxPool2D(pool_size = (2,2),strides = (2,2)))

model.add(Conv2D(filters = 512, kernel_size = (3,3), padding = "same", activation = "relu"))
model.add(Conv2D(filters = 512, kernel_size = (3,3), padding = "same", activation = "relu"))
model.add(Conv2D(filters = 512, kernel_size = (3,3), padding = "same", activation = "relu"))
model.add(MaxPool2D(pool_size = (2,2),strides = (2,2)))

model.add(Conv2D(filters = 512, kernel_size = (3,3), padding = "same", activation = "relu"))
model.add(Conv2D(filters = 512, kernel_size = (3,3), padding = "same", activation = "relu"))
model.add(Conv2D(filters = 512, kernel_size = (3,3), padding = "same", activation = "relu"))
model.add(MaxPool2D(pool_size = (2,2),strides = (2,2)))

model.add(Flatten())
model.add(Dense(units = 4096,activation = "relu"))
model.add(Dense(units = 4096,activation = "relu"))
model.add(Dense(units = 2, activation = "softmax"))

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

### Paso 4: Optimización y Entrenamiento

In [ ]:
checkpoint = ModelCheckpoint("../models/vgg16_best.keras", monitor='val_accuracy', verbose=1, save_best_only=True, mode='max')
early = EarlyStopping(monitor='val_accuracy', min_delta=0, patience=20, verbose=1, mode='max')

hist = model.fit(trdata, validation_data=tsdata, epochs=100, callbacks=[checkpoint, early])

### Paso 5: Guarda el modelo y realiza predicciones

In [ ]:
from tensorflow.keras.models import load_model

best_model = load_model("../models/vgg16_best.keras")
results = best_model.evaluate(X_test, y_test)
print(f"Pérdida en Test: {results[0]}, Precisión en Test: {results[1]}")

preds = best_model.predict(X_test[:5])
for i, p in enumerate(preds):
    label = "Perro" if np.argmax(p) == 1 else "Gato"
    print(f"Imagen {i}: Predicción: {label}")